In [1]:
import numpy as np
import sklearn
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow import keras
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix
import imageio.v3 as iio
import os
from PIL import Image
import imagehash
import shutil
from sklearn.model_selection import train_test_split


# Data loading and Processing

In [2]:
Image_size = 96
Batch = 32
Random_seed = 42

In [3]:
def remove_duplicates(paths, labels):
    unique_hashes = {}
    clean_paths, clean_labels = [], []
    
    print(f"Initial files: {len(paths)}")
    for p, l in zip(paths, labels):
        try:
            with Image.open(p) as img:
                h = str(imagehash.phash(img))
                if h not in unique_hashes:
                    unique_hashes[h] = p
                    clean_paths.append(p)
                    clean_labels.append(l)
        except Exception as e:
            continue
            
    print(f"After dedup: {len(clean_paths)}")
    return clean_paths, clean_labels

# STEP 1: Collect all paths + labels
all_p, all_l = [], []
base_dir = "Training_Tri/"
class_names = sorted(os.listdir(base_dir))
class_to_idx = {name: i for i, name in enumerate(class_names)}

for class_name in class_names:
    c_path = os.path.join(base_dir, class_name)
    if not os.path.isdir(c_path): continue
    files = [os.path.join(c_path, f) for f in os.listdir(c_path)]
    all_p.extend(files)
    all_l.extend([class_to_idx[class_name]] * len(files))

# STEP 2: Deduplicate the WHOLE dataset BEFORE splitting
all_p_clean, all_l_clean = remove_duplicates(all_p, all_l)

# STEP 3: Split the clean pool
tr_p, temp_p, tr_l, temp_l = train_test_split(
    all_p_clean, all_l_clean, test_size=0.3, random_state=Random_seed, stratify=all_l_clean
)
val_p, te_p, val_l, te_l = train_test_split(
    temp_p, temp_l, test_size=0.5, random_state=Random_seed, stratify=temp_l
)

print(f"Split sizes -> Train: {len(tr_p)} | Val: {len(val_p)} | Test: {len(te_p)}")
def load_and_preprocess(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, [Image_size, Image_size])
    return img, label

Initial files: 39375
After dedup: 9050
Split sizes -> Train: 6335 | Val: 1357 | Test: 1358


In [4]:
def check_leakage_by_class(tr_p, tr_l, te_p, te_l):
    train_hashes = {}
    for p, l in zip(tr_p, tr_l):
        with Image.open(p) as img:
            h = str(imagehash.phash(img))
            train_hashes[h] = l

    for cls in np.unique(te_l):
        cls_paths = [p for p, l in zip(te_p, te_l) if l == cls]
        leak = 0
        for p in cls_paths:
            with Image.open(p) as img:
                h = str(imagehash.phash(img))
                if h in train_hashes:
                    leak += 1
        print(f"Class {cls}: {leak}/{len(cls_paths)} leaked ({100*leak/len(cls_paths):.1f}%)")

check_leakage_by_class(tr_p, tr_l, te_p, te_l)

Class 0: 0/774 leaked (0.0%)
Class 1: 0/402 leaked (0.0%)
Class 2: 0/182 leaked (0.0%)


In [5]:
def focal_loss_sparse(gamma, alpha):
    def loss(y_true, y_pred):
        y_true = tf.cast(y_true, tf.int32)
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1 - 1e-7)
        
        # One-hot encode y_true
        y_true_one_hot = tf.one_hot(y_true, depth=y_pred.shape[-1])
        
        # Cross entropy
        ce = -tf.reduce_sum(y_true_one_hot * tf.math.log(y_pred), axis=-1)
        
        # Focal weight
        p_t = tf.reduce_sum(y_true_one_hot * y_pred, axis=-1)
        focal_weight = alpha * tf.pow(1 - p_t, gamma)
        
        return tf.reduce_mean(focal_weight * ce)
    return loss

def apply_blur(img):
    img = tf.expand_dims(img, 0)  
    img = tf.nn.avg_pool2d(img, ksize=3, strides=1, padding='SAME')
    return tf.squeeze(img, 0) 

def augment(image, label):
    image = tf.cast(image, tf.float32) / 255.0
    image = tf.image.random_flip_left_right(image)
    image = tf.image.random_flip_up_down(image)
    image = tf.image.rot90(image, k=tf.random.uniform(shape=[], minval=0, maxval=4, dtype=tf.int32))    
    image = tf.image.random_brightness(image, max_delta=0.3)
    image = tf.image.random_contrast(image, lower=0.7, upper=1.4)
    image = tf.image.random_saturation(image, lower=0.6, upper=1.6)
    image = tf.image.random_hue(image, max_delta=0.05)

    # Haze
    haze_intensity = tf.random.uniform((), 0.05, 0.25)
    haze = tf.ones_like(image) * 0.7
    image = tf.cond(
        tf.random.uniform(()) > 0.5,
        lambda: image * (1 - haze_intensity) + haze * haze_intensity,
        lambda: image
    )

    image = tf.clip_by_value(image, 0.0, 1.0)
    return image, label

def normalize(image, label):
    image = tf.cast(image, tf.float32) / 255.0
    return image, label

def compute_class_weights_Tri(ds):
    labels = np.concatenate([y.numpy() for _, y in ds.batch(512)])    
    classes = np.unique(labels)
    weights = compute_class_weight(class_weight='balanced', classes=classes, y=labels)
    class_weights = dict(zip(classes, weights))
    print(f"Class weights: {class_weights}")
    return class_weights

In [6]:
train_ds = (
    tf.data.Dataset.from_tensor_slices((tr_p, tr_l))
    .shuffle(len(tr_p), seed=Random_seed)
    .map(load_and_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    .map(augment, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(Batch)
    .prefetch(tf.data.AUTOTUNE)
)

val_ds = (
    tf.data.Dataset.from_tensor_slices((val_p, val_l))
    .map(load_and_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    .map(normalize, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(Batch)
    .cache()
    .prefetch(tf.data.AUTOTUNE)
)

test_ds = (
    tf.data.Dataset.from_tensor_slices((te_p, te_l))
    .map(load_and_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    .map(normalize, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(Batch)
    .cache()
    .prefetch(tf.data.AUTOTUNE)
)

# For class weights, build a plain dataset from tr_p/tr_l (no augment)
train_raw_for_weights = (
    tf.data.Dataset.from_tensor_slices((tr_p, tr_l))
    .map(load_and_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
)
class_weights = compute_class_weights_Tri(train_raw_for_weights)

Class weights: {np.int32(0): np.float64(0.5849492151431209), np.int32(1): np.float64(1.1268231945926717), np.int32(2): np.float64(2.481394437916177)}


In [13]:

model = keras.models.Sequential([
    
    keras.layers.Conv2D(8, 3, activation='relu', padding='same'),
    keras.layers.Conv2D(8, 3, activation='relu', padding='same'),
    keras.layers.MaxPooling2D(),
    keras.layers.Dropout(0.2),
    
    keras.layers.Conv2D(16, 3, activation='relu', padding='same'),
    keras.layers.Conv2D(16, 3, activation='relu', padding='same'),
    keras.layers.MaxPooling2D(),
    keras.layers.Dropout(0.2),
   
    keras.layers.Conv2D(32, 3, activation='relu', padding='same'),
    keras.layers.Conv2D(32, 3, activation='relu', padding='same'),
    keras.layers.MaxPooling2D(),
    keras.layers.Dropout(0.3),
   
    keras.layers.Conv2D(64, 3, activation='relu', padding='same'),
    keras.layers.GlobalAveragePooling2D(),

    keras.layers.Dense(16, activation='relu'),
    keras.layers.Dropout(0.2),
    keras.layers.Dense(3, activation='softmax')
])



model.compile(
    optimizer=keras.optimizers.Adam(1e-4),
loss = 'sparse_categorical_crossentropy',
    metrics=['accuracy']
)

callbacks = [
    keras.callbacks.EarlyStopping(
        monitor='val_loss',  # something that actually exists
        patience=5,
        restore_best_weights=True
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=2,
        min_lr=1e-7
    ),
    keras.callbacks.ModelCheckpoint(
        'best_fire_model.keras',
        monitor='val_loss',
        save_best_only=True,
        mode='min'
    )
]

history = model.fit(
    train_ds,
    epochs=30,
    validation_data=val_ds,
    class_weight= class_weights,
    callbacks=callbacks
)

Epoch 1/30


198/198 ━━━━━━━━━━━━━━━━━━━━ 17s 56ms/step - accuracy: 0.3937 - loss: 1.0733 - val_accuracy: 0.4230 - val_loss: 0.8795 - learning_rate: 1.0000e-04
Epoch 2/30
198/198 ━━━━━━━━━━━━━━━━━━━━ 11s 53ms/step - accuracy: 0.6485 - loss: 0.8013 - val_accuracy: 0.8394 - val_loss: 0.5414 - learning_rate: 1.0000e-04
Epoch 3/30
198/198 ━━━━━━━━━━━━━━━━━━━━ 10s 50ms/step - accuracy: 0.6732 - loss: 0.6281 - val_accuracy: 0.8710 - val_loss: 0.4973 - learning_rate: 1.0000e-04
Epoch 4/30
198/198 ━━━━━━━━━━━━━━━━━━━━ 11s 57ms/step - accuracy: 0.7192 - loss: 0.5621 - val_accuracy: 0.5962 - val_loss: 0.5270 - learning_rate: 1.0000e-04
Epoch 5/30
198/198 ━━━━━━━━━━━━━━━━━━━━ 11s 56ms/step - accuracy: 0.7169 - loss: 0.5673 - val_accuracy: 0.8556 - val_loss: 0.4975 - learning_rate: 1.0000e-04
Epoch 6/30
198/198 ━━━━━━━━━━━━━━━━━━━━ 11s 53ms/step - accuracy: 0.7174 - loss: 0.5581 - val_accuracy: 0.7996 - val_loss: 0.4890 - learning_rate: 5.0000e-05
Epoch 7/30
198/198 ━━━━━━━━━━━━━━━━━━━━ 11s 57ms/step - accurac

In [14]:
true_labels = []
y_pred_prob = []

for images, labels in test_ds:
    preds = model(images, training=False)
    true_labels.extend(labels.numpy())
    y_pred_prob.extend(preds.numpy())

true_labels = np.array(true_labels)
y_pred_prob = np.array(y_pred_prob)
pred_classes = np.argmax(y_pred_prob, axis=1)

loss, acc = model.evaluate(test_ds)
print(f"Test Loss:      {loss:.4f}")
print(f"Test Accuracy:  {acc:.4f}")
# Now your confusion matrix and report will be accurate
cm = confusion_matrix(true_labels, pred_classes)
print("\nConfusion Matrix:")
print(cm)

print("\nClassification Report:")
print(classification_report(
    true_labels,
    pred_classes,
))

43/43 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.8657 - loss: 0.3673
Test Loss:      0.3628
Test Accuracy:  0.8682

Confusion Matrix:
[[646   0 128]
 [  0 402   0]
 [ 49   2 131]]

Classification Report:
              precision    recall  f1-score   support

           0       0.93      0.83      0.88       774
           1       1.00      1.00      1.00       402
           2       0.51      0.72      0.59       182

    accuracy                           0.87      1358
   macro avg       0.81      0.85      0.82      1358
weighted avg       0.89      0.87      0.88      1358



In [9]:
def check_leakage_by_class(tr_p, tr_l, te_p, te_l):
    train_hashes = {}
    for p, l in zip(tr_p, tr_l):
        with Image.open(p) as img:
            h = str(imagehash.phash(img))
            train_hashes[h] = l

    for cls in np.unique(te_l):
        cls_paths = [p for p, l in zip(te_p, te_l) if l == cls]
        leak = 0
        for p in cls_paths:
            with Image.open(p) as img:
                h = str(imagehash.phash(img))
                if h in train_hashes:
                    leak += 1
        print(f"Class {cls}: {leak}/{len(cls_paths)} leaked ({100*leak/len(cls_paths):.1f}%)")

check_leakage_by_class(tr_p, tr_l, te_p, te_l)

Class 0: 0/774 leaked (0.0%)
Class 1: 0/402 leaked (0.0%)
Class 2: 0/182 leaked (0.0%)
